In [1]:
import json
import dataclasses
from dataclasses import dataclass
from time import time

import pandas as pd
from kafka import KafkaProducer

In [2]:
@dataclass
class GreenTrip:
    lpep_pickup_datetime: str
    lpep_dropoff_datetime: str
    PULocationID: int
    DOLocationID: int
    passenger_count: int
    trip_distance: float
    tip_amount: float
    total_amount: float


def trip_from_row(row):
    return GreenTrip(
        lpep_pickup_datetime=str(row['lpep_pickup_datetime']),
        lpep_dropoff_datetime=str(row['lpep_dropoff_datetime']),
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        passenger_count=int(row['passenger_count']),
        trip_distance=float(row['trip_distance']),
        tip_amount=float(row['tip_amount']),
        total_amount=float(row['total_amount']),
    )


def trip_serializer(trip):
    trip_dict = dataclasses.asdict(trip)
    return json.dumps(trip_dict).encode('utf-8')

def trip_deserializer(data):
    json_str = data.decode('utf-8')
    trip_dict = json.loads(json_str)
    return GreenTrip(**trip_dict)

In [3]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"

columns = [
    'lpep_pickup_datetime', 
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount'
]

df = pd.read_parquet(url, columns=columns)

# 2. Clean up BEFORE serialization # Filling NaNs so the int() conversion doesn't fail 
df['passenger_count'] = df['passenger_count'].fillna(0).astype(int) 
df['PULocationID'] = df['PULocationID'].fillna(0).astype(int) 
df['DOLocationID'] = df['DOLocationID'].fillna(0).astype(int)

df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1,4.07,6.82,34.12


In [4]:
server = "localhost:9092"

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=trip_serializer
)

topic_name = "green-trips"

In [5]:
t0 = time()

for _, row in df.iterrows():
    trip = trip_from_row(row)
    producer.send(topic_name, value=trip)

producer.flush()

t1 = time()

print(f'took {(t1 - t0):.2f} seconds')

took 9.86 seconds
